In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    df = pd.read_sql("SELECT * FROM songs ORDER BY plays DESC", conn)
    print(df)

In [ ]:
query = """
WITH genre_stats AS (
    SELECT 
        artists.genre,
        songs.title,
        songs.plays,
        SUM(songs.plays) OVER (PARTITION BY artists.genre) as genre_total_plays,
        RANK() OVER (PARTITION BY artists.genre ORDER BY songs.plays DESC) as rank_in_genre
    FROM songs
    JOIN artists ON songs.artist = artists.name
)
SELECT *
FROM genre_stats
WHERE rank_in_genre = 1
ORDER BY genre_total_plays DESC
"""

with engine.connect() as conn:
    genre_df = pd.read_sql(query, conn)

print(genre_df)

In [ ]:
# Load all three tables into pandas and do analysis in Python
with engine.connect() as conn:
    songs_df = pd.read_sql("SELECT * FROM songs", conn)
    artists_df = pd.read_sql("SELECT * FROM artists", conn)

# Merge in pandas - same as SQL JOIN
merged = songs_df.merge(artists_df, left_on="artist", right_on="name")
# left_table.merge(right_table, left_on="column_in_left", right_on="column_in_right")

# Group be genre
genre_summary = merged.groupby("genre")["plays"].agg(
    total_plays="sum", # create total_plays, apply sum
    avg_plays="mean", # create avg_plays, apply mean
    song_count="count" # create song_count, apply count
).sort_values("total_plays", ascending=False)

print(genre_summary)

In [ ]:
with engine.connect() as conn:
    songs_df = pd.read_sql("SELECT * FROM songs", conn)
    artists_df = pd.read_sql("SELECT * FROM artists", conn)
    albums_df = pd.read_sql("SELECT * FROM albums", conn)

# Rename before merging to avoid confusion
artists_df = artists_df.rename(columns={
    "id": "artist_id",
    "name": "artist_name"
})

# Step 1 - songs + artists, print columns to verify
songs_artists = songs_df.merge(
    artists_df,
    left_on="artist",
    right_on="artist_name",
    how="left"
)
print("After first merge:")
print(songs_artists.columns.tolist())

# Step 2 - add albums
full_df = songs_artists.merge(
    albums_df,
    left_on="artist_id",
    right_on="artist_id",
    how="left"
)
print("\nAfter second merge:")
print(full_df.columns.tolist())

# Select and rename meaningful columns
result = full_df[[
    "title_x", "plays", "genre", 
    "country", "title_y", "release_year"
]]
result.columns = ["song", "plays", "genre", "country", "album", "album_year"]

# Check for missing values
print("\nMissing values per column:")
print(result.isnull().sum())

print("\nFull result:")
print(result.sort_values("plays", ascending=False).to_string())

In [ ]:
with engine.connect() as conn:
    query = """
        SELECT 
            s.title as song,
            s.artist,
            s.plays
        FROM songs s
        LEFT JOIN artists ar ON s.artist = ar.name
        LEFT JOIN albums al ON ar.id = al.artist_id
        WHERE al.id IS NULL
        ORDER BY s.plays DESC
    """
    missing_albums = pd.read_sql(query, conn)

print("Songs with no album data:")
print(missing_albums)

In [ ]:
query = """
SELECT title, artist, plays
FROM songs
WHERE plays > (SELECT AVG(plays) FROM songs)
ORDER BY plays DESC
"""

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print("Songs above average plays:")
print(df)

In [ ]:
query = """
SELECT artist, avg_plays
FROM (
    SELECT artist, 
           AVG(plays) AS avg_plays
    FROM songs
    GROUP BY artist
) AS artist_averages
WHERE avg_plays > 1000
ORDER BY avg_plays DESC
"""

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print("Row count:", len(df))
print()
print(df)

In [ ]:
query = """
SELECT s.title,
       s.artist,
       s.plays
FROM songs s
WHERE EXISTS (
    SELECT 1
    FROM artists ar
    WHERE ar.name = s.artist
    AND ar.genre IS NOT NULL
)
ORDER BY s.plays DESC
"""

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print("Row count:", len(df))
print(df)